# ROSP-7 diagnostic: system DIQ built up from components, compared with the direct estimate

Companion to `rosp-07_sdiq_from_atm_seeing.ipynb` for PTH Tier 1 metric
[ROSP-7](https://rubinobs.atlassian.net/browse/ROSP-7). That notebook estimates the system contribution to the
delivered image quality *directly*, as the delivered PSF FWHM minus an atmospheric seeing monitor in quadrature.
This one **builds the system contribution up from its components** and compares the two, in aggregate and
visit by visit.

**Components** (added in quadrature; more may be added later). Everything is in the observed frame of each visit:
no zenith or wavelength correction is applied to the delivered PSF, the AOS or donut terms, and the seeing
monitors are *de-corrected* from zenith / 500 nm to the visit airmass and band exactly as in the companion notebook.

1. **Camera**: a constant $\theta_\mathrm{cam}$ = `camera_fwhm_arcsec` (0.2″) for every visit.
2. **Optical figure**: `aos_fwhm` from `visit1_quicklook` (`optical_figure_source = "aos"`). As a check it is
   compared with the quadrature difference of the delivered PSF and the donut blur,
   $\theta_\mathrm{opt,alt}^2 = \mathrm{FWHM}^2 - \mathrm{donut\_blur}^2$, which can also be selected as the source.
3. **Dome + mirror seeing**: the donut blur minus the atmosphere, for each of the two atmospheric metrics of the
   companion notebook (DIMM with the $L_0$ = 30 m outer-scale correction at wind ≥ 6 m/s, and RINGSS scintillation):
   $$ \theta_\mathrm{dome}^2 = \mathrm{donut\_blur}^2 - \theta_\mathrm{atm,obs}^2 - \theta_\mathrm{cam}^2 $$
   The donut blur contains the camera term (as in `image_quality_trending`), so it is removed here and added back
   once as its own component; `subtract_camera_from_donut_blur = False` turns that off. The optical-figure check
   above needs no camera term because it is in both the delivered PSF and the donut blur and cancels.

The donut blur is not usable in the **y band**, so y-band visits are excluded from everything here (`exclude_bands`).

**Per visit**

$$ \mathrm{sys}^2_\mathrm{comp} = \theta_\mathrm{opt}^2 + \theta_\mathrm{dome}^2 + \theta_\mathrm{cam}^2, \qquad
   \mathrm{sys}^2_\mathrm{direct} = \mathrm{FWHM}^2 - \theta_\mathrm{atm,obs}^2 $$

Every number quoted for a set of visits is the **square root of the median of the squared quantity** over that
set (negative medians are shown as a signed square root), with the robust error of the median propagated through.
A component's own value is $\sqrt{\operatorname{median}(\theta^2)}$ for the same reason. Because the median of a
sum is not the sum of the medians, the headline table gives both the quadrature sum of the component values and
the per-visit sum medianed.

**Algebra worth keeping in mind.** With the defaults the per-visit difference
$\mathrm{sys}^2_\mathrm{comp} - \mathrm{sys}^2_\mathrm{direct} = \mathrm{aos\_fwhm}^2 + \mathrm{donut\_blur}^2
- \mathrm{FWHM}^2$ does not involve the atmospheric monitor (nor the camera) at all; the two metrics then differ
only through which visits each one can use (the DIMM metric drops low-wind visits) and through the dome term,
which does depend on the monitor.

**Parameters** (first code cell): the dayObs range, science programs, the DIMM outer scale and minimum wind, the
RINGSS window, the camera FWHM, the optical-figure source and the EFD name.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
start_dayobs = 20251025  # First dayObs (inclusive)
end_dayobs = 20260715  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>
dimm_outer_scale_m = 30  # outer scale L0 (m) for the von Karman DIMM correction
dimm_min_wind_speed = (
    6.0  # DIMM-based terms only use visits with wind_speed >= this (m/s)
)
ringss_window_minutes = (
    5  # RINGSS measurements within +/- this of the visit midpoint are medianed
)
camera_fwhm_arcsec = 0.2  # constant camera contribution, added in quadrature
optical_figure_source = (
    "aos"  # "aos" (aos_fwhm) or "psf_minus_donut" (sqrt(FWHM^2 - donut_blur^2))
)
subtract_camera_from_donut_blur = (
    True  # the donut blur contains the camera term; remove it when forming dome^2
)
sdiq_goal_arcsec = 0.45  # reference line: nominal system-contribution goal used in image_quality_trending
efd_name = "usdf_efd"  # lsst_efd_client instance name
use_cache = True  # cache the ConsDB/EFD pulls as parquet under ./cache (ignored on Times Square)

not_times_square = True

In [ ]:
# Times Square runs this notebook in a Nublado pod where rubin_nights may be
# missing or stale; install/upgrade it there. Anywhere else this cell is a no-op.
try:
    not_times_square
except NameError:
    not_times_square = False

if not not_times_square:
    print("updating rubin_nights")
    !pip install --user --upgrade git+https://github.com/lsst-sims/rubin_nights.git --no-deps > /dev/null 2>&1

In [ ]:
import inspect
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from astropy.time import Time

%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()

## Constants, parameter helpers, data access

These cells are identical to the companion notebook (the hub rule is no imports between notebooks), so the two
share the same ConsDB query, RINGSS matching and parquet cache files.

In [ ]:
DEFAULT_PROGRAMS = (
    "BLOCK-407",
    "BLOCK-408",
    "BLOCK-416",
    "BLOCK-417",
    "BLOCK-419",
    "BLOCK-421",
)

# Bump when the ConsDB query changes so parquet caches written with the old column set are not reused.
CACHE_VERSION = "v2"

SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
ARCSEC_TO_RAD = np.pi / (180.0 * 3600.0)
REFERENCE_WAVELENGTH_NM = 500.0
CAM_FWHM = 0.207  # arcsec, camera diffusion term used in image_quality_trending (reference line only)

# Effective wavelengths used by lsst.summit.utils.getBandpassSeeingCorrection for LSSTCam.
LSSTCAM_EFFECTIVE_WAVELENGTH_NM = {
    "u": 365.1397,
    "g": 475.6104,
    "r": 618.6055,
    "i": 753.5490,
    "z": 868.0935,
    "y": 972.5979,
}
BAND_ORDER = ("u", "g", "r", "i", "z", "y")

# Columns pulled from cdb_lsstcam.visit1 (per-visit metadata) and visit1_quicklook (image quality).
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "target_name",
    "exp_time",
    "exp_midpt",
    "exp_midpt_mjd",
    "airmass",
    "altitude",
    "azimuth",
    "zenith_distance",
    "air_temp",
    "pressure",
    "humidity",
    "wind_speed",
    "wind_dir",
    "dimm_seeing",
)
QUICKLOOK_COLUMNS = (
    "psf_sigma_median",
    "psf_sigma_min",
    "psf_sigma_max",
    "n_psf_star_total",
    "aos_fwhm",
    "donut_blur_fwhm",
    "ringss_seeing",
)

# Columns pulled from cdb_lsstcam.exposure (joined on exposure_id = visit_id, the 1-to-1 visit system).
EXPOSURE_COLUMNS = ("observation_reason",)

# RINGSS event in the EFD.
RINGSS_TOPIC = "lsst.sal.ESS.logevent_ringssMeasurement"
RINGSS_SAL_INDEX = 304
RINGSS_FIELDS = (
    "timestamp",
    "hrNum",
    "zenithDistance",
    "flux",
    "fwhmScintillation",
    "fwhmSector",
    "fwhmFree",
    "wind",
)

# Chart palette: fixed categorical order (blue, orange, aqua, yellow, magenta, green, violet, red).
SERIES_COLORS = (
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#4a3aa7",
    "#e34948",
)
SERIES_MARKERS = ("o", "s", "^", "D", "v", "P", "X", "*")
INK, INK_SECONDARY, INK_MUTED, GRID, SURFACE = (
    "#0b0b0b",
    "#52514e",
    "#898781",
    "#e1e0d9",
    "#fcfcfb",
)
GOAL_COLOR = "#1baf7a"
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue",
    [
        "#ffffff",
        "#cde2fb",
        "#9ec5f4",
        "#6da7ec",
        "#3987e5",
        "#256abf",
        "#1c5cab",
        "#104281",
        "#0d366b",
    ],
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.labelcolor": INK,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "lines.linewidth": 2.0,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    """Coerce 20260101, "20260101" or "2026-01-01" to an int."""
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """List or comma/space separated string of block names; bare numbers become BLOCK-<n>."""
    if value is None:
        return list(DEFAULT_PROGRAMS)
    items = (
        [v for v in re.split(r"[,\s]+", value.strip()) if v]
        if isinstance(value, str)
        else [str(v).strip() for v in value]
    )
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def day_obs_chunks(day_obs_min: int, day_obs_max: int, chunk_days: int):
    """Yield (lo, hi) day_obs pairs covering the range in calendar chunks."""
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def day_obs_window(day_obs_lo: int, day_obs_hi: int):
    """UTC start/end (astropy Time) covering the nights day_obs_lo..day_obs_hi: 12:00 UTC to 12:00 UTC next day."""
    t0 = datetime.strptime(str(day_obs_lo), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    )
    t1 = datetime.strptime(str(day_obs_hi), "%Y%m%d").replace(
        hour=12, tzinfo=timezone.utc
    ) + pd.Timedelta(days=1)
    return Time(t0), Time(t1)


def _read_cache(cache_path, use_cache):
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    return None


def _write_cache(cache_path, df):
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
        print(f"Saved {len(df)} rows to {cache_path}")
    return df


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, loader())


async def cached_async(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Same as cached() for a loader that returns a coroutine (the EFD client is async on the RSP)."""
    df = _read_cache(cache_path, use_cache)
    return df if df is not None else _write_cache(cache_path, await loader())


COMPONENT_COLORS = {
    "cam": INK_MUTED,
    "opt": "#4a3aa7",
    "dome": "#1baf7a",
    "direct": INK,
}

In [ ]:
USDF_CONSDB_URL = "http://consdb-pq.consdb:8080/consdb"


class _SummitUtilsClient:
    """Adapter so lsst.summit.utils.ConsDbClient.query returns a DataFrame."""

    def __init__(self, url=USDF_CONSDB_URL):
        from lsst.summit.utils import ConsDbClient

        os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
        self._client = ConsDbClient(url)

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    """Object with .query(sql) -> DataFrame: rubin_nights first, then summit_utils."""
    errors = []
    try:
        from rubin_nights import connections

        client = connections.get_clients()["consdb"]
        print("ConsDB client: rubin_nights")
        return client
    except Exception as exc:
        errors.append(f"rubin_nights: {exc!r}")
    try:
        client = _SummitUtilsClient()
        print("ConsDB client: lsst.summit.utils")
        return client
    except Exception as exc:
        errors.append(f"summit_utils: {exc!r}")
    raise RuntimeError("Could not build a ConsDB client:\n  " + "\n  ".join(errors))


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def build_query(
    day_obs_min,
    day_obs_max,
    programs=DEFAULT_PROGRAMS,
    instrument="lsstcam",
    img_types=("science",),
):
    """SQL joining visit1 (pointing, weather, DIMM), visit1_quicklook (PSF stats) and exposure (observation_reason), one row per visit."""
    schema = f"cdb_{instrument}"
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(
    client, day_obs_min, day_obs_max, programs=DEFAULT_PROGRAMS, chunk_days=31
):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    if not frames:
        return pd.DataFrame(
            columns=list(VISIT_COLUMNS)
            + list(QUICKLOOK_COLUMNS)
            + list(EXPOSURE_COLUMNS)
        )
    out = pd.concat(frames, ignore_index=True).sort_values(["day_obs", "seq_num"])
    missing = [
        c for c in VISIT_COLUMNS + QUICKLOOK_COLUMNS + EXPOSURE_COLUMNS if c not in out
    ]
    if missing:
        print(f"WARNING: columns not returned by the ConsDB: {missing}")
    for c in out.columns:
        if (
            c
            not in (
                "band",
                "physical_filter",
                "science_program",
                "observation_reason",
                "img_type",
                "target_name",
                "exp_midpt",
            )
            and out[c].dtype == object
        ):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.reset_index(drop=True)

In [ ]:
def get_efd_client(efd_name="usdf_efd"):
    import lsst_efd_client

    return lsst_efd_client.EfdClient(efd_name, output_mode="dataframe")


async def fetch_ringss(
    efd_client, day_obs_min, day_obs_max, chunk_days=31, sal_index=RINGSS_SAL_INDEX
):
    """All RINGSS measurements between 12:00 UTC on day_obs_min and 12:00 UTC the day after day_obs_max."""
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        start, end = day_obs_window(lo, hi)
        df = efd_client.select_time_series(
            RINGSS_TOPIC, list(RINGSS_FIELDS), start, end, index=sal_index
        )
        if inspect.isawaitable(df):
            df = await df
        print(f"  ringss {lo}-{hi}: {len(df):6d} measurements")
        if len(df):
            df = df.reset_index().rename(columns={df.index.name or "index": "time"})
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["time", *RINGSS_FIELDS])
    out = pd.concat(frames, ignore_index=True)
    out["time"] = pd.to_datetime(out["time"], utc=True).dt.tz_localize(None)
    return out.sort_values("time").reset_index(drop=True)


async def load_ringss(
    efd_name, day_obs_min, day_obs_max, cache_path=None, use_cache=True
):
    """RINGSS table from the EFD, or None (with a warning) if the EFD is unavailable."""
    try:
        efd = get_efd_client(efd_name)
        ringss = await cached_async(
            cache_path, lambda: fetch_ringss(efd, day_obs_min, day_obs_max), use_cache
        )
    except Exception as exc:
        print(
            f"WARNING: RINGSS pull from the EFD failed ({exc!r}); "
            "falling back to the ConsDB ringss_seeing column."
        )
        return None
    if ringss is None or len(ringss) == 0:
        print(
            "WARNING: the EFD returned no RINGSS measurements; "
            "falling back to the ConsDB ringss_seeing column."
        )
        return None
    return ringss


def match_ringss_to_visits(visits, ringss, window_minutes=5.0):
    """Median RINGSS quantities within +/- window of each visit midpoint.

    Returns a copy of ``visits`` with ringss_scint, ringss_sector, ringss_free, ringss_wind,
    ringss_zd, ringss_n and ringss_source ("efd" or "consdb"). With ``ringss=None`` the ConsDB
    ``ringss_seeing`` column stands in for ringss_scint.
    """
    out = visits.copy()
    cols = {
        "ringss_scint": "fwhmScintillation",
        "ringss_sector": "fwhmSector",
        "ringss_free": "fwhmFree",
        "ringss_wind": "wind",
        "ringss_zd": "zenithDistance",
    }
    for c in cols:
        out[c] = np.nan
    out["ringss_n"] = 0
    if ringss is None or len(ringss) == 0:
        out["ringss_scint"] = pd.to_numeric(out["ringss_seeing"], errors="coerce")
        out["ringss_n"] = out["ringss_scint"].notna().astype(int)
        out["ringss_source"] = "consdb"
        return out
    out["ringss_source"] = "efd"
    r = ringss.dropna(subset=["time"]).sort_values("time")
    t_r = r["time"].to_numpy(dtype="datetime64[ns]").astype("int64") / 1e9
    vals = {
        c: r[src].to_numpy(dtype=float) if src in r else np.full(len(r), np.nan)
        for c, src in cols.items()
    }
    t_v = (
        pd.to_datetime(out["exp_midpt"], utc=True, format="ISO8601")
        .dt.tz_localize(None)
        .to_numpy(dtype="datetime64[ns]")
        .astype("int64")
        / 1e9
    )
    w = window_minutes * 60.0
    lo = np.searchsorted(t_r, t_v - w, side="left")
    hi = np.searchsorted(t_r, t_v + w, side="right")
    res = {c: np.full(len(out), np.nan) for c in cols}
    n = np.zeros(len(out), dtype=int)
    for i, (a, b) in enumerate(zip(lo, hi)):
        if b > a:
            n[i] = b - a
            for c in cols:
                seg = vals[c][a:b]
                seg = seg[np.isfinite(seg)]
                if len(seg):
                    res[c][i] = np.median(seg)
    for c in cols:
        out[c] = res[c]
    out["ringss_n"] = n
    return out

## Physics: atmosphere de-correction, direct sys², and the components

`add_derived_columns` (shared with the companion notebook) gives the observed FWHM, each atmospheric metric at
zenith / 500 nm (`<tag>_z500`) and de-corrected to the visit (`atm_<tag>`), and the direct `sys2_<tag>`.
`add_component_columns` then adds, per visit, `opt2_aos`, `opt2_psf_donut`, `opt2` (the chosen source),
`dome2_<tag>`, `comp2_<tag>` = opt² + dome² + cam², and `diff2_<tag>` = comp² − direct sys².

In [ ]:
def airmass_seeing_correction(airmass):
    """Multiply an observed FWHM by this to refer it to zenith (X^-0.6)."""
    airmass = np.asarray(airmass, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(airmass >= 1.0, airmass ** (-0.6), np.nan)


def bandpass_seeing_correction(band):
    """Multiply an observed FWHM by this to refer it to 500 nm; accepts 'r' or 'r_57'."""
    first = pd.Series(band).astype("string").str.strip().str[0]
    wl = first.map(LSSTCAM_EFFECTIVE_WAVELENGTH_NM).astype(float)
    return (wl.to_numpy() / REFERENCE_WAVELENGTH_NM) ** 0.2


def seeing_zenith_500nm_to_observed(seeing_z500, airmass, band):
    """De-correct a zenith / 500 nm seeing to the airmass and band of a visit: X^0.6 (lambda/500nm)^-0.2."""
    seeing = np.asarray(seeing_z500, dtype=float)
    return (
        seeing / airmass_seeing_correction(airmass) / bandpass_seeing_correction(band)
    )


def fried_parameter(seeing_arcsec, wavelength_nm=REFERENCE_WAVELENGTH_NM):
    """r0 (m) from a Kolmogorov FWHM in arcsec: r0 = 0.98 lambda / FWHM."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return (
            0.98
            * wavelength_nm
            * 1e-9
            / (np.asarray(seeing_arcsec, dtype=float) * ARCSEC_TO_RAD)
        )


def outer_scale_correction(seeing_arcsec, L0):
    """sqrt(1 - 2.18 (r0/L0)^0.356) (Tokovinin 2002); 1 for L0=None; NaN where the bracket is negative."""
    seeing = np.asarray(seeing_arcsec, dtype=float)
    if L0 is None:
        return np.where(np.isfinite(seeing), 1.0, np.nan)
    with np.errstate(invalid="ignore"):
        bracket = 1.0 - 2.18 * np.power(fried_parameter(seeing) / float(L0), 0.356)
        return np.where(bracket > 0, np.sqrt(np.clip(bracket, 0, None)), np.nan)


def signed_sqrt(x):
    x = np.asarray(x, dtype=float)
    return np.sign(x) * np.sqrt(np.abs(x))


@dataclass
class AtmMetric:
    tag: str
    label: str
    short: str
    color: str
    marker: str

    @property
    def z500(self):
        return f"{self.tag}_z500"

    @property
    def atm(self):
        return f"atm_{self.tag}"

    @property
    def sys2(self):
        return f"sys2_{self.tag}"


def make_metrics(L0, dimm_min_wind):
    return [
        AtmMetric(
            "dimm",
            f"DIMM, L0 = {L0:g} m (wind >= {dimm_min_wind:g} m/s)",
            "DIMM L0",
            SERIES_COLORS[0],
            SERIES_MARKERS[0],
        ),
        AtmMetric(
            "ringss",
            "RINGSS scintillation",
            "RINGSS",
            SERIES_COLORS[1],
            SERIES_MARKERS[1],
        ),
    ]


def add_derived_columns(df, metrics, L0, dimm_min_wind):
    """Observed FWHM, each metric at zenith/500nm and in the observed frame, and sys2 per metric."""
    out = df.copy()
    out["psf_fwhm"] = out["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
    out["fwhm_zenith_500nm"] = (
        out["psf_fwhm"]
        * airmass_seeing_correction(out["airmass"])
        * bandpass_seeing_correction(out["band"])
    )
    out["dimm_r0"] = fried_parameter(out["dimm_seeing"])
    out["dimm_z500"] = out["dimm_seeing"].to_numpy(
        dtype=float
    ) * outer_scale_correction(out["dimm_seeing"], L0)
    out["dimm_windy"] = out["wind_speed"] >= dimm_min_wind
    out.loc[~out["dimm_windy"].fillna(False), "dimm_z500"] = np.nan
    out["ringss_z500"] = out["ringss_scint"].astype(float)
    for m in metrics:
        out[m.atm] = seeing_zenith_500nm_to_observed(
            out[m.z500], out["airmass"], out["band"]
        )
        out[m.sys2] = out["psf_fwhm"] ** 2 - out[m.atm] ** 2
    out["time"] = pd.to_datetime(
        out["exp_midpt"], utc=True, errors="coerce"
    ).dt.tz_localize(None)
    out["date"] = pd.to_datetime(out["day_obs"].astype(int).astype(str))
    out["rel_wind_az"] = (
        (out["azimuth"] - out["wind_dir"] + 180.0) % 360.0
    ) - 180.0  # 0 = into the wind
    out["abs_rel_wind_az"] = out["rel_wind_az"].abs()
    return out


OPTICAL_FIGURE_SOURCES = {
    "aos": ("opt2_aos", "aos_fwhm"),
    "psf_minus_donut": ("opt2_psf_donut", "sqrt(FWHM² - donut_blur²)"),
}


def add_component_columns(
    df, metrics, camera_fwhm, optical_source="aos", subtract_camera=False
):
    out = df.copy()
    out["cam2"] = float(camera_fwhm) ** 2
    out["donut2"] = out["donut_blur_fwhm"].astype(float) ** 2
    out["opt2_aos"] = out["aos_fwhm"].astype(float) ** 2
    out["opt2_psf_donut"] = out["psf_fwhm"] ** 2 - out["donut2"]
    if optical_source not in OPTICAL_FIGURE_SOURCES:
        raise ValueError(
            f"optical_figure_source must be one of {list(OPTICAL_FIGURE_SOURCES)}"
        )
    out["opt2"] = out[OPTICAL_FIGURE_SOURCES[optical_source][0]]
    for m in metrics:
        dome2 = out["donut2"] - out[m.atm] ** 2
        if subtract_camera:
            dome2 = dome2 - out["cam2"]
        out[f"dome2_{m.tag}"] = dome2
        out[f"comp2_{m.tag}"] = out["opt2"] + dome2 + out["cam2"]
        out[f"diff2_{m.tag}"] = out[f"comp2_{m.tag}"] - out[m.sys2]
    return out


@dataclass
class QualityCuts:
    fwhm_range: tuple = (0.4, 4.0)
    donut_range: tuple = (0.2, 4.0)  # donut_blur_fwhm must be present and in this range
    aos_range: tuple = (0.0, 3.0)  # aos_fwhm must be present and in this range
    dimm_range: tuple = (0.3, 4.0)
    ringss_range: tuple = (0.2, 4.0)
    airmass_max: float = 2.5
    exclude_bands: tuple = ("y",)  # the donut blur is not usable in y
    exclude_reasons: tuple = (
        "filter_change_close_loop",
    )  # substring match on observation_reason
    require_both: bool = (
        False  # True: keep only visits where both atmospheric metrics are valid
    )


def apply_quality_cuts(df, metrics, cuts=QualityCuts(), verbose=True):
    """Drop visits without usable PSF / donut / AOS / wind values; blank each metric where its monitor is invalid."""
    steps = [
        ("has PSF FWHM", df["psf_fwhm"].notna()),
        (f"PSF FWHM in {cuts.fwhm_range}", df["psf_fwhm"].between(*cuts.fwhm_range)),
        (
            f"donut_blur_fwhm in {cuts.donut_range}",
            df["donut_blur_fwhm"].between(*cuts.donut_range),
        ),
        (f"aos_fwhm in {cuts.aos_range}", df["aos_fwhm"].between(*cuts.aos_range)),
        (f"airmass <= {cuts.airmass_max}", df["airmass"] <= cuts.airmass_max),
        ("has wind speed", df["wind_speed"].notna()),
    ]
    if cuts.exclude_bands:
        steps.append(
            (f"band not in {cuts.exclude_bands}", ~df["band"].isin(cuts.exclude_bands))
        )
    if "observation_reason" in df:
        for reason in cuts.exclude_reasons:
            steps.append(
                (
                    f"observation_reason without '{reason}'",
                    ~df["observation_reason"]
                    .astype("string")
                    .fillna("")
                    .str.contains(reason),
                )
            )
    elif cuts.exclude_reasons:
        print(
            "NOTE: no observation_reason column (stale cache or query change); "
            f"the {cuts.exclude_reasons} cut is skipped."
        )
    dimm_ok = df["dimm_seeing"].between(*cuts.dimm_range) & df["dimm_z500"].notna()
    ringss_ok = df["ringss_scint"].between(*cuts.ringss_range)
    if cuts.require_both:
        steps.append(("DIMM and RINGSS both valid", dimm_ok & ringss_ok))
    else:
        steps.append(("DIMM or RINGSS valid", dimm_ok | ringss_ok))
    keep = pd.Series(True, index=df.index)
    rows = []
    for name, mask in steps:
        mask = mask.fillna(False).astype(bool)
        rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
        keep &= mask
    out = df[keep].copy()
    by_tag = {"dimm": dimm_ok, "ringss": ringss_ok}
    for m in metrics:
        bad = ~by_tag[m.tag][keep].fillna(False).astype(bool)
        out.loc[
            bad,
            [
                m.z500,
                m.atm,
                m.sys2,
                f"dome2_{m.tag}",
                f"comp2_{m.tag}",
                f"diff2_{m.tag}",
            ],
        ] = np.nan
    if verbose:
        print(f"Start: {len(df)} visits")
        print(
            pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(
                index=False
            )
        )
        print("\nvisits with a valid atmospheric metric:")
        for m in metrics:
            print(f"  {m.label}: {int(out[m.sys2].notna().sum())}")
    return out

## Statistics and plotting helpers

`sdiq_stats` / `binned_sdiq` are the shared estimator (median of a squared quantity, robust error, signed sqrt).
`agg_sqrt_median` applies it to any squared column; `nightly_table` / `plot_nightly_panels` give the
10th / 50th / 90th percentile night-by-night view of any set of squared columns; `plot_quantity_vs` is the
cross-term figure (hexbin per metric, binned sqrt-of-median, counts) for any squared column against any x.

In [ ]:
def sdiq_stats(sys2, min_count=1):
    """Median of sys2 (arcsec^2), robust SE of the median, and the signed-sqrt sDIQ with propagated error."""
    s = np.asarray(sys2, dtype=float)
    s = s[np.isfinite(s)]
    row = dict(
        n=len(s),
        median_sys2=np.nan,
        median_sys2_err=np.nan,
        p16_sys2=np.nan,
        p84_sys2=np.nan,
        frac_negative=np.nan,
        sdiq=np.nan,
        sdiq_err=np.nan,
    )
    if len(s) < min_count or len(s) == 0:
        return row
    med = np.median(s)
    mad = np.median(np.abs(s - med))
    err = 1.2533 * 1.4826 * mad / np.sqrt(len(s))
    row.update(
        median_sys2=med,
        median_sys2_err=err,
        p16_sys2=np.percentile(s, 16),
        p84_sys2=np.percentile(s, 84),
        frac_negative=float((s < 0).mean()),
        sdiq=float(signed_sqrt(med)),
        sdiq_err=err / (2.0 * np.sqrt(abs(med))) if med != 0 else np.nan,
    )
    return row


def binned_sdiq(x, sys2, bins, min_count=10):
    """sdiq_stats of sys2 in bins of x; rows with n < min_count are left NaN."""
    x, s = np.asarray(x, dtype=float), np.asarray(sys2, dtype=float)
    ok = np.isfinite(x) & np.isfinite(s)
    x, s = x[ok], s[ok]
    edges = np.asarray(bins, dtype=float)
    idx = np.digitize(x, edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        sel = s[idx == i]
        row = dict(lo=edges[i], hi=edges[i + 1], center=0.5 * (edges[i] + edges[i + 1]))
        row.update(sdiq_stats(sel, min_count=min_count))
        rows.append(row)
    return pd.DataFrame(rows)


def _break_gaps(nightly, max_gap_days=3):
    """Insert NaN rows after gaps longer than max_gap_days so lines break instead of bridging."""
    nightly = nightly.sort_values("date").reset_index(drop=True)
    gaps = nightly["date"].diff().dt.days > max_gap_days
    if not gaps.any():
        return nightly
    pieces = []
    for i, (_, row) in enumerate(nightly.iterrows()):
        if gaps.iloc[i]:
            blank = row.copy()
            blank[:] = np.nan
            blank["date"] = row["date"] - pd.Timedelta(days=1)
            pieces.append(blank)
        pieces.append(row)
    return pd.DataFrame(pieces).reset_index(drop=True)


def _percentile_series(
    ax, x, med, p10, p90, color, marker, label, offset_days=0.0, ms=5
):
    """IQT-style night marker at the median with a thin bar from the 10th to the 90th percentile."""
    x = pd.to_datetime(x) + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        med,
        yerr=np.array([med - p10, p90 - med]),
        fmt="none",
        ecolor=color,
        elinewidth=1.0,
        alpha=0.6,
        capsize=0,
    )
    ax.plot(x, med, marker=marker, ms=ms, lw=1.0, color=color, label=label)


def agg_sqrt_median(df, col, min_count=1):
    """sdiq_stats of a squared column, renamed: n, median2, median2_err, value (= signed sqrt), value_err, p16/p84, frac_negative."""
    st = sdiq_stats(df[col], min_count=min_count)
    return dict(
        n=st["n"],
        median2=st["median_sys2"],
        median2_err=st["median_sys2_err"],
        value=st["sdiq"],
        value_err=st["sdiq_err"],
        p16_2=st["p16_sys2"],
        p84_2=st["p84_sys2"],
        frac_negative=st["frac_negative"],
    )


def component_table(df, metrics, camera_fwhm):
    """Per metric, on that metric's valid visits: each component, their quadrature sum, the per-visit sum medianed,
    and the direct estimate."""
    rows = []
    for m in metrics:
        sub = df[df[f"comp2_{m.tag}"].notna() & df[m.sys2].notna()]
        opt = agg_sqrt_median(sub, "opt2")
        dome = agg_sqrt_median(sub, f"dome2_{m.tag}")
        comp = agg_sqrt_median(sub, f"comp2_{m.tag}")
        direct = agg_sqrt_median(sub, m.sys2)
        diff = agg_sqrt_median(sub, f"diff2_{m.tag}")
        quad = opt["median2"] + dome["median2"] + camera_fwhm**2
        rows.append(
            dict(
                metric=m.label,
                n_visits=len(sub),
                n_nights=int(sub["day_obs"].nunique()),
                camera=camera_fwhm,
                optical_figure=opt["value"],
                optical_figure_err=opt["value_err"],
                dome_mirror=dome["value"],
                dome_mirror_err=dome["value_err"],
                quad_sum_of_components=float(signed_sqrt(quad)),
                components_sum_then_median=comp["value"],
                components_sum_then_median_err=comp["value_err"],
                direct_sdiq=direct["value"],
                direct_sdiq_err=direct["value_err"],
                median_diff2=diff["median2"],
                median_diff2_err=diff["median2_err"],
                frac_comp_above_direct=float((sub[f"diff2_{m.tag}"] > 0).mean()),
            )
        )
    return pd.DataFrame(rows).set_index("metric")


def table_by_group(df, metrics, group_col, cols, min_count=10):
    """sqrt-of-median of several squared columns per group (band, ...); cols = [(name, column or {tag: column})]."""
    rows = []
    for key, sub in df.groupby(group_col):
        row = {group_col: key, "n_visits": len(sub)}
        for name, col in cols:
            if isinstance(col, dict):
                for m in metrics:
                    st = agg_sqrt_median(sub, col[m.tag], min_count=min_count)
                    row[f"{name}_{m.tag}"] = st["value"]
            else:
                st = agg_sqrt_median(sub, col, min_count=min_count)
                row[name] = st["value"]
        rows.append(row)
    out = pd.DataFrame(rows).set_index(group_col)
    if group_col == "band":
        out = out.reindex([b for b in BAND_ORDER if b in out.index])
    return out


def nightly_table(df, specs, min_count=5):
    """One row per night. specs = [(name, column, squared)]: squared columns give the signed sqrt of the
    10th/50th/90th percentiles of the column (so the median is sqrt(median)) plus the propagated error;
    plain columns give their own percentiles."""

    def pct(x, q):
        x = np.asarray(x, dtype=float)
        x = x[np.isfinite(x)]
        return np.percentile(x, q) if len(x) else np.nan

    rows = []
    for day_obs, sub in df.groupby("day_obs"):
        row = dict(
            day_obs=int(day_obs),
            date=pd.Timestamp(str(int(day_obs))),
            n_visits=len(sub),
            wind_speed=sub["wind_speed"].median(),
        )
        for name, col, squared in specs:
            x = sub[col]
            n = int(x.notna().sum())
            row[f"n_{name}"] = n
            if n < min_count or n == 0:
                (
                    row[name],
                    row[f"{name}_p10"],
                    row[f"{name}_p90"],
                    row[f"{name}_err"],
                ) = (np.nan,) * 4
                continue
            if squared:
                st = sdiq_stats(x)
                row[name] = st["sdiq"]
                row[f"{name}_err"] = st["sdiq_err"]
                row[f"{name}_p10"] = float(signed_sqrt(pct(x, 10)))
                row[f"{name}_p90"] = float(signed_sqrt(pct(x, 90)))
            else:
                row[name] = pct(x, 50)
                row[f"{name}_err"] = np.nan
                row[f"{name}_p10"] = pct(x, 10)
                row[f"{name}_p90"] = pct(x, 90)
        rows.append(row)
    return pd.DataFrame(rows)


def plot_nightly_panels(nightly, panels, goal=None, height_per_panel=3.0):
    """IQT-style nightly figure. panels = [dict(series=[(name, label, color, marker)], ylabel, goal=bool)],
    plus a visits / wind panel at the bottom. Markers: nightly median (sqrt of median for squared columns);
    thin bars: 10th-90th percentile; thick capped bars (where available): error on the median.
    """
    nightly = _break_gaps(nightly)
    n = len(panels)
    fig, axes = plt.subplots(
        n + 1,
        1,
        figsize=(13, height_per_panel * n + 2.2),
        sharex=True,
        gridspec_kw={"height_ratios": [1.0] * n + [0.45]},
    )
    for ax, panel in zip(axes[:-1], panels):
        series = panel["series"]
        offsets = np.linspace(-0.22, 0.22, len(series)) if len(series) > 1 else [0.0]
        for (name, label, color, marker), off in zip(series, offsets):
            _percentile_series(
                ax,
                nightly["date"],
                nightly[name],
                nightly[f"{name}_p10"],
                nightly[f"{name}_p90"],
                color,
                marker,
                label,
                off,
            )
            err = nightly.get(f"{name}_err")
            if err is not None and np.isfinite(err).any():
                ax.errorbar(
                    pd.to_datetime(nightly["date"]) + pd.Timedelta(days=off),
                    nightly[name],
                    yerr=err,
                    fmt="none",
                    ecolor=color,
                    elinewidth=2.5,
                    capsize=3,
                )
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        if goal and panel.get("goal"):
            ax.axhline(goal, color=GOAL_COLOR, lw=1.5, ls="--", label=f'goal {goal:g}"')
            ax.axhspan(0, goal, color=GOAL_COLOR, alpha=0.08, lw=0)
        ax.set_ylabel(panel["ylabel"])
        lo, hi = ax.get_ylim()  # leave room above the bars for the legend
        ax.set_ylim(lo, hi + 0.45 * (hi - lo))
        ax.legend(
            ncol=3,
            fontsize=9,
            loc="upper left",
            title=panel.get(
                "title",
                "marker: nightly median; thin bar: 10th-90th pct; thick bar: error on the median",
            ),
            title_fontsize=9,
        )
    ax = axes[-1]
    ax.bar(nightly["date"], nightly["n_visits"], width=0.8, color=GRID)
    ax.set_ylabel("visits")
    ax2 = ax.twinx()
    ax2.plot(
        nightly["date"],
        nightly["wind_speed"],
        marker="o",
        ms=3,
        lw=1.2,
        color=SERIES_COLORS[2],
    )
    ax2.set_ylabel("median wind (m/s)", color=SERIES_COLORS[2])
    ax2.grid(False)
    ax.set_xlabel("night (day_obs)")
    fig.align_ylabels(axes)
    return fig


def plot_squared_histograms(
    df, items, bins=np.arange(-1.0, 2.0, 0.025), xlabel="arcsec²", goal=None, title=None
):
    """Step histograms of squared columns. items = [(column, label, color[, linestyle])]; medians marked and quoted as sqrt."""
    fig, ax = plt.subplots(figsize=(9, 5))
    for item in items:
        col, label, color = item[:3]
        ls = item[3] if len(item) > 3 else "-"
        s = df[col].dropna()
        if not len(s):
            continue
        st = sdiq_stats(s)
        ax.hist(
            s.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=2,
            color=color,
            ls=ls,
            label=f'{label}: median {st["median_sys2"]:+.3f} arcsec² -> {st["sdiq"]:.3f}" (n = {st["n"]})',
        )
        ax.axvline(st["median_sys2"], color=color, lw=1.2, ls=":")
    ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    if goal:
        ax.axvline(
            goal**2,
            color=GOAL_COLOR,
            lw=1.5,
            ls="--",
            label=f'goal {goal:g}" (= {goal**2:.3f} arcsec²)',
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("visits")
    ax.set_xlim(bins[0], bins[-1])
    ax.legend(loc="upper left", fontsize=9)
    if title:
        ax.set_title(title)
    return fig


def plot_pair_hexbin(df, pairs, gridsize=45, ncols=2):
    """Hexbin of y vs x per (xcol, ycol, xlabel, ylabel, title, lim) with the 1:1 line and binned median (of y in x bins)."""
    n = len(pairs)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(5.4 * ncols, 5.0 * nrows),
        squeeze=False,
        layout="constrained",
    )
    hb = None
    for ax, (xcol, ycol, xlabel, ylabel, title, lim) in zip(axes.flat, pairs):
        ok = df[xcol].notna() & df[ycol].notna()
        x, y = df.loc[ok, xcol], df.loc[ok, ycol]
        if ok.sum():
            hb = ax.hexbin(
                x,
                y,
                gridsize=gridsize,
                extent=(*lim, *lim),
                cmap=SEQUENTIAL_BLUE,
                norm=LogNorm(vmin=1),
                mincnt=1,
                linewidths=0.2,
                edgecolors="face",
            )
        xbins = np.linspace(lim[0], lim[1], 23)
        st = binned_sdiq(
            x, y, xbins, min_count=10
        )  # median of y per x bin (sdiq column = signed sqrt, median_sys2 = median)
        ax.plot(lim, lim, color=INK_MUTED, lw=1, ls="--", label="1:1")
        ax.plot(st["center"], st["median_sys2"], color=SURFACE, lw=4)
        ax.plot(
            st["center"],
            st["median_sys2"],
            color=SERIES_COLORS[1],
            lw=2,
            marker="o",
            ms=4,
            label="binned median",
        )
        ax.set_title(f"{title}   (n = {int(ok.sum())})", fontsize=11)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_xlim(*lim)
        ax.set_ylim(*lim)
        ax.set_aspect("equal")
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    axes[0, 0].legend(loc="upper left", fontsize=9)
    if hb is not None:
        cbar = fig.colorbar(
            hb, ax=[a for a in axes.flat if a.get_visible()], pad=0.02, shrink=0.9
        )
        cbar.set_label("visits per hexagon")
    return fig


def plot_quantity_vs(
    df,
    metrics,
    ycols,
    xcol,
    bins,
    xlabel,
    ylabel_sq="arcsec²",
    ylabel_sqrt="sqrt(binned median) (arcsec)",
    ylim_sq=(-0.8, 1.6),
    ylim_sqrt=None,
    gridsize=40,
    min_count=10,
    goal=None,
    ref_lines=(),
    title=None,
    mask=None,
):
    """A squared quantity vs x: hexbin per metric with the binned median (top), its sqrt for all metrics (middle), counts.

    ``ycols`` / ``xcol`` may be column names or dicts metric.tag -> column name.
    ``ref_lines`` is a list of (xgrid, ygrid, label) reference curves for the sqrt panel.
    """
    n = len(metrics)
    edges = np.asarray(bins, dtype=float)
    xmin, xmax = float(edges[0]), float(edges[-1])
    fig = plt.figure(figsize=(5.4 * n, 11.5), layout="constrained")
    gs = fig.add_gridspec(3, n, height_ratios=[3.2, 3.2, 1.1])
    hex_axes = [fig.add_subplot(gs[0, k]) for k in range(n)]
    ax_sq = fig.add_subplot(gs[1, :])
    ax_cnt = fig.add_subplot(gs[2, :], sharex=ax_sq)
    hb = None
    tables = {}
    sel_by_tag = {}
    for ax, m in zip(hex_axes, metrics):
        xc = xcol[m.tag] if isinstance(xcol, dict) else xcol
        yc = ycols[m.tag] if isinstance(ycols, dict) else ycols
        ok = df[yc].notna() & df[xc].notna()
        if mask is not None:
            ok &= mask
        sel_by_tag[m.tag] = (xc, ok)
        x, y = df.loc[ok, xc], df.loc[ok, yc]
        if ok.sum():
            hb = ax.hexbin(
                x,
                y,
                gridsize=gridsize,
                extent=(xmin, xmax, *ylim_sq),
                cmap=SEQUENTIAL_BLUE,
                norm=LogNorm(vmin=1),
                mincnt=1,
                linewidths=0.2,
                edgecolors="face",
            )
        st = binned_sdiq(x, y, edges, min_count=min_count)
        tables[m.tag] = st
        ax.fill_between(
            st["center"],
            st["p16_sys2"],
            st["p84_sys2"],
            color=m.color,
            alpha=0.15,
            lw=0,
        )
        ax.plot(
            st["center"], st["median_sys2"], color=SURFACE, lw=4, solid_capstyle="round"
        )
        ax.plot(
            st["center"],
            st["median_sys2"],
            color=m.color,
            lw=2,
            marker="o",
            ms=4,
            label="binned median (16-84 % band)",
        )
        ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_title(f"{m.label}   (n = {int(ok.sum())})", fontsize=11)
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(*ylim_sq)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel_sq)
    hex_axes[0].legend(loc="upper left", fontsize=9)
    if hb is not None:
        cbar = fig.colorbar(hb, ax=hex_axes, pad=0.02, shrink=0.9)
        cbar.set_label("visits per hexagon")
    for m in metrics:
        st = tables[m.tag]
        ax_sq.errorbar(
            st["center"],
            st["sdiq"],
            yerr=st["sdiq_err"],
            color=m.color,
            marker=m.marker,
            ms=6,
            capsize=2,
            lw=2,
            label=m.label,
        )
    for k, (xg, yg, lab) in enumerate(ref_lines):
        ax_sq.plot(
            xg, yg, color=INK_MUTED, lw=1.4, ls=(":", "-.", "--")[k % 3], label=lab
        )
    ax_sq.axhline(0, color=INK_MUTED, lw=1, ls="--")
    if goal:
        ax_sq.axhline(goal, color=GOAL_COLOR, lw=1.5, ls="--", label=f'goal {goal:g}"')
    ax_sq.set_ylabel(ylabel_sqrt)
    if ylim_sqrt:
        ax_sq.set_ylim(*ylim_sqrt)
    ax_sq.set_xlim(xmin, xmax)
    ax_sq.legend(fontsize=9)
    ax_sq.tick_params(labelbottom=False)
    for m in metrics:
        xc, ok = sel_by_tag[m.tag]
        ax_cnt.hist(df.loc[ok, xc], bins=edges, histtype="step", lw=1.8, color=m.color)
    ax_cnt.set_xlabel(xlabel)
    ax_cnt.set_ylabel("visits")
    if title:
        fig.suptitle(title)
    return fig, tables, dict(hex=hex_axes, sqrt=ax_sq, counts=ax_cnt)


def binned_table(tables, metrics, column="sdiq"):
    """Side-by-side binned values (default the signed sqrt of the binned median) for every metric, plus counts."""
    out = pd.DataFrame({"center": next(iter(tables.values()))["center"]}).set_index(
        "center"
    )
    for m in metrics:
        out[f"{column}_{m.tag}"] = tables[m.tag][column].to_numpy()
        out[f"n_{m.tag}"] = tables[m.tag]["n"].to_numpy()
    return out


def plot_component_buildup(table, metrics, camera_fwhm, goal=None):
    """Stacked bars of the median squared components next to the direct median sys² per metric (arcsec²),
    annotated in arcsec."""
    fig, ax = plt.subplots(figsize=(4.0 * len(metrics) + 4, 5.5), layout="constrained")
    width = 0.38
    xticks, xlabels = [], []
    for k, m in enumerate(metrics):
        row = table.loc[m.label]
        x0, x1 = 2 * k, 2 * k + width + 0.08
        parts = [
            ("cam", camera_fwhm**2, f'camera {camera_fwhm:g}"'),
            (
                "opt",
                row["optical_figure"] ** 2 * np.sign(row["optical_figure"]),
                f'optical figure {row["optical_figure"]:.3f}"',
            ),
            (
                "dome",
                row["dome_mirror"] ** 2 * np.sign(row["dome_mirror"]),
                f'dome + mirror {row["dome_mirror"]:.3f}"',
            ),
        ]
        bottom = 0.0
        for key, val, lab in parts:
            ax.bar(
                x0,
                val,
                width,
                bottom=bottom,
                color=COMPONENT_COLORS[key],
                label=lab if k == 0 else None,
                edgecolor=SURFACE,
                lw=0.8,
            )
            bottom += val
        ax.text(
            x0,
            bottom + 0.005,
            f'{row["quad_sum_of_components"]:.3f}"',
            ha="center",
            va="bottom",
            fontsize=9,
        )
        d2 = row["direct_sdiq"] ** 2 * np.sign(row["direct_sdiq"])
        ax.bar(
            x1,
            d2,
            width,
            color=COMPONENT_COLORS["direct"],
            label="direct: FWHM² - atm²" if k == 0 else None,
        )
        ax.errorbar(
            x1,
            d2,
            yerr=2 * abs(row["direct_sdiq"]) * row["direct_sdiq_err"],
            fmt="none",
            ecolor=SURFACE,
            capsize=3,
        )
        ax.text(
            x1,
            d2 + 0.005,
            f'{row["direct_sdiq"]:.3f}"',
            ha="center",
            va="bottom",
            fontsize=9,
        )
        xticks += [x0, x1]
        xlabels += [f"components\n{m.short}", f"direct\n{m.short}"]
    if goal:
        ax.axhline(
            goal**2,
            color=GOAL_COLOR,
            lw=1.5,
            ls="--",
            label=f'goal {goal:g}" (= {goal**2:.3f} arcsec²)',
        )
    ax.set_xticks(xticks, xlabels)
    ax.set_ylabel("median squared contribution (arcsec²)")
    ax.legend(fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    ax.set_title(
        "System DIQ: components (quadrature sum of the component medians) vs the direct estimate"
    )
    return fig

## Parameters and data

In [ ]:
start_dayobs = parse_day_obs(start_dayobs)
end_dayobs = parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
dimm_outer_scale_m = float(dimm_outer_scale_m)
dimm_min_wind_speed = float(dimm_min_wind_speed)
ringss_window_minutes = float(ringss_window_minutes)
camera_fwhm_arcsec = float(camera_fwhm_arcsec)
optical_figure_source = str(optical_figure_source).strip().lower()
subtract_camera_from_donut_blur = parse_bool(subtract_camera_from_donut_blur)
sdiq_goal_arcsec = (
    float(sdiq_goal_arcsec) if sdiq_goal_arcsec not in (None, "", "none") else None
)
use_cache = parse_bool(use_cache)

METRICS = make_metrics(dimm_outer_scale_m, dimm_min_wind_speed)
DIMM, RINGSS = METRICS
OPT_LABEL = OPTICAL_FIGURE_SOURCES[optical_figure_source][1]
DOME_LABEL = "donut_blur² - atm²" + (
    " - cam²" if subtract_camera_from_donut_blur else ""
)

cache_tag = f"{CACHE_VERSION}_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs)}"
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)

print(f"Nights {start_dayobs}-{end_dayobs}; programs {programs}")
print("Atmospheric metrics:", [m.label for m in METRICS])
print(
    f'Components: camera {camera_fwhm_arcsec:g}", optical figure = {OPT_LABEL}, dome + mirror = {DOME_LABEL}'
)

In [ ]:
consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
ringss = await load_ringss(
    efd_name,
    start_dayobs,
    end_dayobs,
    cache_dir
    and os.path.join(cache_dir, f"ringss_{start_dayobs}_{end_dayobs}.parquet"),
    use_cache,
)
visits = match_ringss_to_visits(raw, ringss, ringss_window_minutes)

print(
    f"{len(visits)} visits on {visits['day_obs'].nunique()} nights; RINGSS source: "
    f"{visits['ringss_source'].iloc[0] if len(visits) else 'n/a'}"
)
visits.head()

In [ ]:
# What came back, per program and band, and how complete the key columns are.
if len(visits):
    print(pd.crosstab(visits["science_program"], visits["band"], margins=True))
print("\nfraction of visits with a value:")
print(
    visits[
        [
            "psf_sigma_median",
            "aos_fwhm",
            "donut_blur_fwhm",
            "dimm_seeing",
            "ringss_scint",
            "wind_speed",
            "wind_dir",
        ]
    ]
    .notna()
    .mean()
    .round(3)
)
print("\nfraction with a donut blur, per band (expect ~0 in y):")
print(
    visits.groupby("band")["donut_blur_fwhm"].apply(lambda s: s.notna().mean()).round(3)
)

## Derived quantities and quality cuts

Beyond the companion notebook's cuts (PSF range, airmass, wind reading, AOS-loop-closing visits), every visit
here needs a donut blur and an `aos_fwhm` in range, and the **y band is excluded** because the donut blur is not
usable there. Each atmospheric metric is then valid on its own sample (DIMM: reading in range and wind ≥
`dimm_min_wind_speed`; RINGSS: reading in range); set `require_both=True` to compare them on identical visits.

In [ ]:
visits = add_derived_columns(visits, METRICS, dimm_outer_scale_m, dimm_min_wind_speed)
visits = add_component_columns(
    visits,
    METRICS,
    camera_fwhm_arcsec,
    optical_figure_source,
    subtract_camera_from_donut_blur,
)

cuts = QualityCuts(
    fwhm_range=(0.4, 4.0),
    donut_range=(0.2, 4.0),
    aos_range=(0.0, 3.0),
    dimm_range=(0.3, 4.0),
    ringss_range=(0.2, 4.0),
    airmass_max=2.5,
    exclude_bands=("y",),
    exclude_reasons=("filter_change_close_loop",),
    require_both=False,
)
sel = apply_quality_cuts(visits, METRICS, cuts)

## 1. Optical figure: `aos_fwhm` versus the delivered PSF minus the donut blur

If the delivered PSF were just the optical figure on top of the donut blur, $\sqrt{\mathrm{FWHM}^2 -
\mathrm{donut\_blur}^2}$ would reproduce `aos_fwhm`. This section needs no atmospheric monitor, so it uses every
visit that passed the cuts. The headline compares $\sqrt{\operatorname{median}(\mathrm{aos\_fwhm}^2)}$ with
$\sqrt{\operatorname{median}(\mathrm{FWHM}^2 - \mathrm{donut\_blur}^2)}$, then per band, per night, and visit by visit.

In [ ]:
opt_rows = {
    "aos_fwhm": agg_sqrt_median(sel, "opt2_aos"),
    "sqrt(FWHM² - donut_blur²)": agg_sqrt_median(sel, "opt2_psf_donut"),
}
sel["optdiff2"] = sel["opt2_psf_donut"] - sel["opt2_aos"]
opt_rows["difference (psf-donut minus aos, in quadrature)"] = agg_sqrt_median(
    sel, "optdiff2"
)
opt_table = pd.DataFrame(opt_rows).T
for name in list(opt_rows)[:2]:
    r = opt_table.loc[name]
    print(
        f'{name}: {r["value"]:.3f} +/- {r["value_err"]:.3f} arcsec (n = {int(r["n"])})'
    )
opt_table.round(4)

In [ ]:
table_by_group(
    sel,
    METRICS,
    "band",
    [
        ("aos_fwhm", "opt2_aos"),
        ("psf_minus_donut", "opt2_psf_donut"),
        ("difference", "optdiff2"),
    ],
).round(3)

In [ ]:
fig = plot_squared_histograms(
    sel,
    [
        ("opt2_aos", "aos_fwhm²", COMPONENT_COLORS["opt"]),
        ("opt2_psf_donut", "FWHM² - donut_blur²", SERIES_COLORS[1]),
        ("optdiff2", "(FWHM² - donut_blur²) - aos_fwhm²", INK_MUTED),
    ],
    bins=np.arange(-0.6, 1.2, 0.02),
    xlabel="squared optical-figure estimate (arcsec²)",
    title="optical figure: the two estimates and their difference",
)
plt.show()

sel["opt_psf_donut"] = signed_sqrt(sel["opt2_psf_donut"])
fig = plot_pair_hexbin(
    sel,
    [
        (
            "aos_fwhm",
            "opt_psf_donut",
            "aos_fwhm (arcsec)",
            "signed sqrt(FWHM² - donut_blur²) (arcsec)",
            "per visit",
            (-0.2, 1.2),
        ),
        (
            "opt2_aos",
            "opt2_psf_donut",
            "aos_fwhm² (arcsec²)",
            "FWHM² - donut_blur² (arcsec²)",
            "per visit, squared",
            (-0.2, 0.8),
        ),
    ],
)
plt.show()

In [ ]:
nightly_opt = nightly_table(
    sel,
    [
        ("aos", "opt2_aos", True),
        ("psf_donut", "opt2_psf_donut", True),
        ("optdiff", "optdiff2", True),
    ],
    min_count=5,
)
fig = plot_nightly_panels(
    nightly_opt,
    [
        dict(
            series=[
                ("aos", "aos_fwhm", COMPONENT_COLORS["opt"], "o"),
                ("psf_donut", "sqrt(FWHM² - donut_blur²)", SERIES_COLORS[1], "s"),
            ],
            ylabel="optical figure (arcsec)",
            goal=False,
        ),
        dict(
            series=[
                (
                    "optdiff",
                    "signed sqrt((FWHM² - donut_blur²) - aos_fwhm²)",
                    INK_MUTED,
                    "D",
                )
            ],
            ylabel="difference (arcsec)",
            goal=False,
        ),
    ],
)
plt.show()

## 2. Dome + mirror seeing: donut blur minus the atmosphere

$\theta_\mathrm{dome}^2 = \mathrm{donut\_blur}^2 - \theta_\mathrm{atm,obs}^2$ (minus the camera if selected) for each
atmospheric metric, quoted as the square root of the median. Being the blur the wavefront sensors see minus what the
outside monitors see, it is the component expected to respond to wind speed, to the pointing relative to the wind
and to the seeing itself.

In [ ]:
dome_rows = {m.label: agg_sqrt_median(sel, f"dome2_{m.tag}") for m in METRICS}
dome_rows["donut_blur (for reference)"] = agg_sqrt_median(sel, "donut2")
dome_table = pd.DataFrame(dome_rows).T
for m in METRICS:
    r = dome_table.loc[m.label]
    print(
        f'dome + mirror [{m.label}]: {r["value"]:.3f} +/- {r["value_err"]:.3f} arcsec (n = {int(r["n"])})'
    )
dome_table.round(4)

In [ ]:
table_by_group(
    sel,
    METRICS,
    "band",
    [("dome", {m.tag: f"dome2_{m.tag}" for m in METRICS}), ("donut_blur", "donut2")],
).round(3)

In [ ]:
fig = plot_squared_histograms(
    sel,
    [(f"dome2_{m.tag}", f"dome + mirror [{m.label}]", m.color) for m in METRICS],
    bins=np.arange(-1.0, 2.0, 0.025),
    xlabel=f"{DOME_LABEL} (arcsec²)",
    title="dome + mirror seeing, squared",
)
plt.show()

In [ ]:
nightly_dome = nightly_table(
    sel,
    [(f"dome_{m.tag}", f"dome2_{m.tag}", True) for m in METRICS]
    + [("donut", "donut2", True)],
    min_count=5,
)
fig = plot_nightly_panels(
    nightly_dome,
    [
        dict(
            series=[("donut", "donut_blur", INK, "o")]
            + [
                (f"dome_{m.tag}", f"dome + mirror [{m.short}]", m.color, m.marker)
                for m in METRICS
            ],
            ylabel="arcsec",
            goal=False,
        ),
    ],
)
plt.show()

In [ ]:
wind_bins = np.arange(0, 20.1, 1.0)
DOME2 = {m.tag: f"dome2_{m.tag}" for m in METRICS}
fig, dome_wind, axes = plot_quantity_vs(
    sel,
    METRICS,
    DOME2,
    "wind_speed",
    wind_bins,
    "wind speed (m/s)",
    ylabel_sq=f"{DOME_LABEL} (arcsec²)",
    ylabel_sqrt="dome + mirror seeing (arcsec)",
    title="dome + mirror seeing vs wind speed",
)
for ax in axes["hex"] + [axes["sqrt"]]:
    ax.axvspan(wind_bins[0], dimm_min_wind_speed, color=INK_MUTED, alpha=0.08, lw=0)
plt.show()
binned_table(dome_wind, METRICS).round(3)

In [ ]:
az_bins = np.arange(-180, 181, 30)
windy = sel["wind_speed"] >= dimm_min_wind_speed
fig, dome_az, axes = plot_quantity_vs(
    sel,
    METRICS,
    DOME2,
    "rel_wind_az",
    az_bins,
    "telescope azimuth - wind direction (deg; 0 = into the wind)",
    ylabel_sq=f"{DOME_LABEL} (arcsec²)",
    ylabel_sqrt="dome + mirror seeing (arcsec)",
    mask=windy,
    title=f"dome + mirror seeing vs pointing relative to the wind (wind >= {dimm_min_wind_speed:g} m/s)",
)
for ax in axes["hex"] + [axes["sqrt"], axes["counts"]]:
    ax.set_xticks(np.arange(-180, 181, 60))
plt.show()
binned_table(dome_az, METRICS).round(3)

In [ ]:
atm_bins = np.arange(0.3, 2.01, 0.1)
fig, dome_atm, _ = plot_quantity_vs(
    sel,
    METRICS,
    DOME2,
    {m.tag: m.atm for m in METRICS},
    atm_bins,
    "atmospheric seeing, de-corrected to visit airmass/band (arcsec)",
    ylabel_sq=f"{DOME_LABEL} (arcsec²)",
    ylabel_sqrt="dome + mirror seeing (arcsec)",
    title="dome + mirror seeing vs the atmospheric seeing metric",
)
plt.show()
binned_table(dome_atm, METRICS).round(3)

## 3. Build-up and comparison with the direct sDIQ

For each atmospheric metric, on the visits where both estimates exist: the three components, their quadrature
sum, the per-visit sum medianed (`components_sum_then_median`, the like-for-like number), and the direct estimate
$\sqrt{\operatorname{median}(\mathrm{FWHM}^2 - \mathrm{atm}^2)}$. `median_diff2` is the median of the per-visit
difference $\mathrm{sys}^2_\mathrm{comp} - \mathrm{sys}^2_\mathrm{direct}$ in arcsec² (positive: the components
over-predict the direct estimate).

In [ ]:
headline = component_table(sel, METRICS, camera_fwhm_arcsec)
for m in METRICS:
    r = headline.loc[m.label]
    print(
        f'{m.label}: components {r["components_sum_then_median"]:.3f} +/- {r["components_sum_then_median_err"]:.3f}"'
        f' (quadrature sum of medians {r["quad_sum_of_components"]:.3f}") vs direct {r["direct_sdiq"]:.3f} +/- {r["direct_sdiq_err"]:.3f}"'
        f'; median diff² {r["median_diff2"]:+.4f} arcsec² (n = {int(r["n_visits"])})'
    )
headline.round(4).T

In [ ]:
fig = plot_component_buildup(
    headline, METRICS, camera_fwhm_arcsec, goal=sdiq_goal_arcsec
)
plt.show()

In [ ]:
# Same per band.
table_by_group(
    sel,
    METRICS,
    "band",
    [
        ("components", {m.tag: f"comp2_{m.tag}" for m in METRICS}),
        ("direct", {m.tag: m.sys2 for m in METRICS}),
    ],
).round(3)

In [ ]:
fig = plot_squared_histograms(
    sel,
    [(f"comp2_{m.tag}", f"components [{m.short}]", m.color) for m in METRICS]
    + [(m.sys2, f"direct [{m.short}]", m.color, "--") for m in METRICS],
    bins=np.arange(-1.0, 2.0, 0.025),
    xlabel="system DIQ squared (arcsec²)",
    goal=sdiq_goal_arcsec,
    title="components (solid) vs direct (dashed), squared",
)
plt.show()

fig = plot_squared_histograms(
    sel,
    [
        (f"diff2_{m.tag}", f"components² - direct² [{m.label}]", m.color)
        for m in METRICS
    ],
    bins=np.arange(-1.0, 1.0, 0.02),
    xlabel="components² - direct² (arcsec²)",
    title="per-visit difference",
)
plt.show()

In [ ]:
# Visit by visit: components vs direct, in arcsec² (the natural space of the subtraction).
fig = plot_pair_hexbin(
    sel,
    [
        (
            m.sys2,
            f"comp2_{m.tag}",
            "direct: FWHM² - atm² (arcsec²)",
            "components: opt² + dome² + cam² (arcsec²)",
            m.label,
            (-0.6, 1.2),
        )
        for m in METRICS
    ],
)
plt.show()

In [ ]:
nightly_sys = nightly_table(
    sel,
    [(f"comp_{m.tag}", f"comp2_{m.tag}", True) for m in METRICS]
    + [(f"direct_{m.tag}", m.sys2, True) for m in METRICS]
    + [(f"diff_{m.tag}", f"diff2_{m.tag}", True) for m in METRICS],
    min_count=5,
)
fig = plot_nightly_panels(
    nightly_sys,
    [
        dict(
            series=[
                (f"comp_{m.tag}", f"components [{m.short}]", m.color, m.marker)
                for m in METRICS
            ]
            + [
                (f"direct_{m.tag}", f"direct [{m.short}]", m.color, "x")
                for m in METRICS
            ],
            ylabel="system DIQ (arcsec)",
            goal=True,
        ),
        dict(
            series=[
                (
                    f"diff_{m.tag}",
                    f"signed sqrt(components² - direct²) [{m.short}]",
                    m.color,
                    m.marker,
                )
                for m in METRICS
            ],
            ylabel="difference (arcsec)",
            goal=False,
        ),
    ],
    goal=sdiq_goal_arcsec,
)
plt.show()

nightly_sys.dropna(subset=["day_obs"]).astype({"day_obs": int}).set_index(
    "day_obs"
).drop(columns="date").round(3)

### Where the two estimates disagree

The per-visit difference $\mathrm{sys}^2_\mathrm{comp} - \mathrm{sys}^2_\mathrm{direct}$ against wind speed,
`aos_fwhm` and the atmospheric seeing. A flat curve at zero means the component model closes; a trend says which
regime one of the two is missing something in.

In [ ]:
DIFF2 = {m.tag: f"diff2_{m.tag}" for m in METRICS}
fig, diff_wind, axes = plot_quantity_vs(
    sel,
    METRICS,
    DIFF2,
    "wind_speed",
    wind_bins,
    "wind speed (m/s)",
    ylim_sq=(-0.8, 0.8),
    ylabel_sq="components² - direct² (arcsec²)",
    ylabel_sqrt="signed sqrt(binned median) (arcsec)",
    title="components - direct vs wind speed",
)
for ax in axes["hex"] + [axes["sqrt"]]:
    ax.axvspan(wind_bins[0], dimm_min_wind_speed, color=INK_MUTED, alpha=0.08, lw=0)
plt.show()
binned_table(diff_wind, METRICS, column="median_sys2").round(4)

In [ ]:
aos_bins = np.arange(0.0, 1.01, 0.05)
fig, diff_aos, _ = plot_quantity_vs(
    sel,
    METRICS,
    DIFF2,
    "aos_fwhm",
    aos_bins,
    "aos_fwhm (arcsec)",
    ylim_sq=(-0.8, 0.8),
    ylabel_sq="components² - direct² (arcsec²)",
    ylabel_sqrt="signed sqrt(binned median) (arcsec)",
    title="components - direct vs aos_fwhm",
)
plt.show()
binned_table(diff_aos, METRICS, column="median_sys2").round(4)

In [ ]:
fig, diff_atm, _ = plot_quantity_vs(
    sel,
    METRICS,
    DIFF2,
    {m.tag: m.atm for m in METRICS},
    atm_bins,
    "atmospheric seeing, de-corrected to visit airmass/band (arcsec)",
    ylim_sq=(-0.8, 0.8),
    ylabel_sq="components² - direct² (arcsec²)",
    ylabel_sqrt="signed sqrt(binned median) (arcsec)",
    title="components - direct vs the atmospheric seeing metric",
)
plt.show()
binned_table(diff_atm, METRICS, column="median_sys2").round(4)

## Notes

* With `optical_figure_source = "psf_minus_donut"` and the camera not subtracted, the components reduce
  algebraically to $\mathrm{FWHM}^2 - \mathrm{atm}^2 + \theta_\mathrm{cam}^2$, i.e. the direct estimate plus the
  camera; that setting is a consistency check, not an independent estimate.
* The y band is excluded throughout because the donut blur is not measured there; the companion notebook's direct
  sDIQ includes y, so its headline numbers are not expected to match this notebook's `direct_sdiq` exactly.
* Both atmospheric monitors sit outside the dome, so the dome + mirror term also absorbs wind shake, tracking and
  any monitor calibration error.
* Changes to the method go through the
  [PTH Method Change Process](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501).